In [1]:
# !pip install pyyaml
import yaml
import pandas as pd

# Load one file to understand the structure
with open('/Users/aadi/documents/ipl_cricsheet/335982.yaml', 'r') as f:
    match = yaml.safe_load(f)

# See the top-level keys
print(match.keys())
print(match['info'].keys())


dict_keys(['meta', 'info', 'innings'])
dict_keys(['balls_per_over', 'city', 'competition', 'dates', 'gender', 'match_type', 'outcome', 'overs', 'player_of_match', 'players', 'registry', 'teams', 'toss', 'umpires', 'venue'])


In [2]:
# match['info']['dates']
match_info = pd.DataFrame()
match_index = 1

# match['info']['match_type_number']
match_info.loc[match_index,'team1'] = match['info']['teams'][0]
match_info.loc[match_index,'team2'] = match['info']['teams'][1]
match_info.loc[match_index,'winner'] = match['info']['outcome']['winner']
# match_info['by'] = match['info']['outcome']['by']
print(match_info)


                         team1                  team2                 winner
1  Royal Challengers Bangalore  Kolkata Knight Riders  Kolkata Knight Riders


In [3]:
import os
import yaml
import pandas as pd

folder_path = '/Users/aadi/documents/ipl_cricsheet/'

# Get all yaml files, sort by the number in the filename
yaml_files = [f for f in os.listdir(folder_path) if f.endswith('.yaml')]
yaml_files.sort(key=lambda f: int(f.replace('.yaml', '')))

rows = []

for i, filename in enumerate(yaml_files, start=1):
    filepath = os.path.join(folder_path, filename)
    with open(filepath, 'r') as f:
        match = yaml.safe_load(f)

    info = match['info']
    toss = info.get('toss', {})
    outcome = info.get('outcome', {})
    umpires = info.get('umpires', [])
    pom = info.get('player_of_match', [])

    # Outcome fields
    winner = outcome.get('winner')
    result = outcome.get('result')
    by = outcome.get('by', {})
    if 'runs' in by:
        win_type, win_margin = 'runs', by['runs']
    elif 'wickets' in by:
        win_type, win_margin = 'wickets', by['wickets']
    else:
        win_type, win_margin = None, None

    # Handle tie with super over / bowl out
    if not winner and result:
        if 'eliminator' in outcome:
            winner = outcome['eliminator']
        elif 'bowl_out' in outcome:
            winner = outcome['bowl_out']

    rows.append({
        'filename': filename,
        'season': str(info.get('season', '')) or None,
        'match_number': (info.get('event') or {}).get('match_number'),
        'date': info['dates'][0],
        'team1': info['teams'][0],
        'team2': info['teams'][1],
        'city': info.get('city'),
        'venue': info.get('venue'),
        'neutral_venue': info.get('neutral_venue'),
        'toss_winner': toss.get('winner'),
        'toss_decision': toss.get('decision'),
        'winner': winner,
        'win_type': win_type,
        'win_margin': win_margin,
        'result': result,
        'method': outcome.get('method'),
        'eliminator': outcome.get('eliminator'),
        'player_of_match': pom[0] if pom else None,
        'umpire1': umpires[0] if len(umpires) > 0 else None,
        'umpire2': umpires[1] if len(umpires) > 1 else None,
    })

    if i % 200 == 0:
        print(f'Processed {i} files...')

match_info = pd.DataFrame(rows)
print(f'Done! Total matches: {len(match_info)}')
match_info.head(10)

Processed 200 files...


Processed 400 files...


Processed 600 files...


Processed 800 files...


Processed 1000 files...


Done! Total matches: 1169


,filename,season,match_number,date,team1,team2,city,venue,neutral_venue,toss_winner,toss_decision,winner,win_type,win_margin,result,method,eliminator,player_of_match,umpire1,umpire2
0,335982.yaml,None,None,2008-04-18,Royal Challengers Bangalore,Kolkata Knight Riders,Bangalore,M Chinnaswamy Stadium,NaN,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,None,None,None,BB McCullum,Asad Rauf,RE Koertzen
1,335983.yaml,None,None,2008-04-19,Kings XI Punjab,Chennai Super Kings,Chandigarh,"Punjab Cricket Association Stadium, Mohali",NaN,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,None,None,None,MEK Hussey,MR Benson,SL Shastri
2,335984.yaml,None,None,2008-04-19,Delhi Daredevils,Rajasthan Royals,Delhi,Feroz Shah Kotla,NaN,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,None,None,None,MF Maharoof,Aleem Dar,GA Pratapkumar
3,335985.yaml,None,None,2008-04-20,Mumbai Indians,Royal Challengers Bangalore,Mumbai,Wankhede Stadium,NaN,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,None,None,None,MV Boucher,SJ Davis,DJ Harper
4,335986.yaml,None,None,2008-04-20,Kolkata Knight Riders,Deccan Chargers,Kolkata,Eden Gardens,NaN,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,None,None,None,DJ Hussey,BF Bowden,K Hariharan
5,335987.yaml,None,None,2008-04-21,Rajasthan Royals,Kings XI Punjab,Jaipur,Sawai Mansingh Stadium,NaN,Kings XI Punjab,bat,Rajasthan Royals,wickets,6.0,None,None,None,SR Watson,Aleem Dar,RB Tiffin
6,335988.yaml,None,None,2008-04-22,Deccan Chargers,Delhi Daredevils,Hyderabad,"Rajiv Gandhi International Stadium, Uppal",NaN,Deccan Chargers,bat,Delhi Daredevils,wickets,9.0,None,None,None,V Sehwag,IL Howell,AM Saheba
7,335989.yaml,None,None,2008-04-23,Chennai Super Kings,Mumbai Indians,Chennai,"MA Chidambaram Stadium, Chepauk",NaN,Mumbai Indians,field,Chennai Super Kings,runs,6.0,None,None,None,ML Hayden,DJ Harper,GA Pratapkumar
8,335990.yaml,None,None,2008-04-24,Deccan Chargers,Rajasthan Royals,Hyderabad,"Rajiv Gandhi International Stadium, Uppal",NaN,Rajasthan Royals,field,Rajasthan Royals,wickets,3.0,None,None,None,YK Pathan,Asad Rauf,MR Benson
9,335991.yaml,None,None,2008-04-25,Kings XI Punjab,Mumbai Indians,Chandigarh,"Punjab Cricket Association Stadium, Mohali",NaN,Mumbai Indians,field,Kings XI Punjab,runs,66.0,None,None,None,KC Sangakkara,Aleem Dar,AM Saheba


In [4]:
match_info.iloc[1167]

filename                               1473511.yaml
season                                         None
match_number                                   None
date                                     2025-06-03
team1                   Royal Challengers Bengaluru
team2                                  Punjab Kings
city                                      Ahmedabad
venue              Narendra Modi Stadium, Ahmedabad
neutral_venue                                   NaN
toss_winner                            Punjab Kings
toss_decision                                 field
winner                  Royal Challengers Bengaluru
win_type                                       runs
win_margin                                      6.0
result                                         None
method                                         None
eliminator                                     None
player_of_match                           KH Pandya
umpire1                               J Madanagopal
umpire2     

In [5]:
match_info[match_info['winner'] == "no result"]

,filename,season,match_number,date,team1,team2,city,venue,neutral_venue,toss_winner,toss_decision,winner,win_type,win_margin,result,method,eliminator,player_of_match,umpire1,umpire2


In [6]:
match_info.to_csv('/Users/aadi/Documents/ipl/data_modeling/match_info.csv', index=False)